# VQE Optimizer Comparison: SPSA vs CMA-ES vs Parameter-Shift

Compares three optimization methods on a ring-topology ZZ Hamiltonian:
- **Parameter-Shift** (exact gradient, 2P evaluations per step)
- **SPSA** (stochastic gradient, 2 evaluations per step)
- **CMA-ES** (population-based, derivative-free)

Tested on two ansatze:
- **QAOA** (problem + mixer layers)
- **HEA** (hardware-efficient ansatz: RY/RZ + CNOT ring)

All using **efficient contraction** (exact expectation values, no shot noise).

**Part 1**: Shallow circuits (N=6,8; L=1,2) — all 3 optimizers.

**Part 2**: Deep/complex circuits (N=6-14; L=2-4; rank=16) — CMA-ES vs ParamShift head-to-head.

## 1. Setup

In [ ]:
!pip install -q torch numpy pandas matplotlib seaborn hashable_list ordered_set
!pip install -q git+https://github.com/keunjunpark/TREV@real_form_autograd

In [ ]:
import math, time, gc
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.optimization.optimizer import Optimizer
from TREV.optimization.optimization import minimize
from TREV.optimization.gradients.batch_parameter_shift import BatchParameterShiftGradient
from TREV.optimization.gradients.spsa import SPSAGradient
from TREV.optimization.cma_es import CMAES, minimize_cma_es

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem_in_bytes / 1e9:.1f} GB')

## 2. Circuit and Hamiltonian Builders

In [ ]:
def ring_zz_hamiltonian(n_qubits):
    """ZZ ring Hamiltonian: sum_i Z_i Z_{(i+1) % N}."""
    terms, coeffs = [], []
    for i in range(n_qubits):
        j = (i + 1) % n_qubits
        pauli = ['I'] * n_qubits
        pauli[i] = 'Z'
        pauli[j] = 'Z'
        terms.append(''.join(pauli))
        coeffs.append(1.0)
    return Hamiltonian(n_qubits, terms, coeffs)


def build_qaoa_circuit(n_qubits, p_layers, rank):
    """QAOA ansatz on ring topology.
    
    Problem layer: CNOT-RZ-CNOT per ring edge (ZZ interaction).
    Mixer layer: RX on each qubit.
    Params per layer: N (RZ edges) + N (RX mixer) = 2N.
    Total params: 2N * p_layers.
    """
    c = Circuit(num_qubit=n_qubits, rank=rank, device=DEVICE)
    for i in range(n_qubits):
        c.h(i)
    for _ in range(p_layers):
        # Problem layer: ZZ via CNOT-RZ-CNOT on ring
        for i in range(n_qubits):
            j = (i + 1) % n_qubits
            c.cx(i, j)
            c.rz(j)
            c.cx(i, j)
        # Mixer layer
        for i in range(n_qubits):
            c.rx(i)
    return c


def build_hea_circuit(n_qubits, layers, rank):
    """Hardware-efficient ansatz on ring topology.
    
    Each layer: CNOT ring + RY + RZ per qubit.
    Params per layer: 2N (RY + RZ).
    Total params: 2N * layers.
    """
    c = Circuit(num_qubit=n_qubits, rank=rank, device=DEVICE)
    for i in range(n_qubits):
        c.h(i)
    for _ in range(layers):
        for i in range(n_qubits):
            c.cx(i, (i + 1) % n_qubits)
        for i in range(n_qubits):
            c.ry(i)
            c.rz(i)
    return c


# Quick test
for name, builder in [('QAOA', build_qaoa_circuit), ('HEA', build_hea_circuit)]:
    c = builder(6, 2, 8)
    print(f'{name}: {c.num_qubit} qubits, {c.params_size} params, {len(c.gates)} gates')

---
# Part 1: Shallow Circuits (All 3 Optimizers)

## 3. Experiment Configuration

In [ ]:
# ── Grid ──
N_QUBITS_LIST = [6, 8]          # number of qubits
LAYERS_LIST   = [1, 2]          # QAOA p / HEA layers
RANK          = 8               # bond dimension
SEED          = 42

# ── Optimizer settings ──
N_ITERS       = 150             # iterations (param-shift, SPSA)
N_GENS        = 150             # generations (CMA-ES)
LR            = 0.05            # learning rate for Adam (param-shift, SPSA)
SPSA_C        = 0.1             # SPSA perturbation size
CMA_SIGMA     = 0.5             # CMA-ES initial step size

# All use exact contraction (shots=0)
MEASURE = MeasureMethod.EFFICIENT_CONTRACTION
SHOTS   = 0

configs = []
for nq in N_QUBITS_LIST:
    for layers in LAYERS_LIST:
        for ansatz in ['QAOA', 'HEA']:
            configs.append((nq, layers, ansatz))

print(f'{len(configs)} circuit configs x 3 optimizers = {len(configs) * 3} runs')

## 4. Run All Experiments

In [ ]:
results = []

for idx, (nq, layers, ansatz) in enumerate(configs):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # Build circuit and Hamiltonian
    if ansatz == 'QAOA':
        circuit = build_qaoa_circuit(nq, layers, RANK)
    else:
        circuit = build_hea_circuit(nq, layers, RANK)

    hamil = ring_zz_hamiltonian(nq)
    P = circuit.params_size
    theta0 = 0.1 * torch.randn(P, device=DEVICE, generator=torch.Generator(device=DEVICE).manual_seed(SEED))

    tag = f'{ansatz} N={nq} L={layers} (P={P})'
    print(f'\n{"="*60}')
    print(f'Config {idx+1}/{len(configs)}: {tag}')
    print(f'{"="*60}')

    # ── 1. Parameter-Shift + Adam ──
    print(f'\n  [1/3] Parameter-Shift ...')
    ps_grad = BatchParameterShiftGradient(
        shift=math.pi / 2, batch_size=None, shots=SHOTS,
        measure_method=MEASURE, depth=1)
    ps_opt = Optimizer(torch.optim.Adam, {'lr': LR})

    t0 = time.time()
    _, ps_exp, _, ps_times = minimize(
        circuit, theta0.clone(), hamil, ps_opt, ps_grad,
        iteration=N_ITERS, best_value_method='highest_probability')
    ps_total = time.time() - t0
    ps_exp = [float(v) for v in ps_exp]

    results.append(dict(
        ansatz=ansatz, n_qubits=nq, layers=layers, params=P,
        optimizer='ParamShift', exp_values=ps_exp,
        iter_times=ps_times, total_time=ps_total))
    print(f'    final={ps_exp[-1]:.4f}  min={min(ps_exp):.4f}  time={ps_total:.1f}s')

    if hasattr(ps_grad, '_gpu_pool') and ps_grad._gpu_pool is not None:
        ps_grad._gpu_pool.shutdown()
    del ps_grad; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    # ── 2. SPSA + Adam ──
    print(f'  [2/3] SPSA ...')
    spsa_grad = SPSAGradient(
        measure_method=MEASURE, shots=SHOTS, c=SPSA_C)
    spsa_opt = Optimizer(torch.optim.Adam, {'lr': LR})

    t0 = time.time()
    _, spsa_exp, _, spsa_times = minimize(
        circuit, theta0.clone(), hamil, spsa_opt, spsa_grad,
        iteration=N_ITERS, best_value_method='highest_probability')
    spsa_total = time.time() - t0
    spsa_exp = [float(v) for v in spsa_exp]

    results.append(dict(
        ansatz=ansatz, n_qubits=nq, layers=layers, params=P,
        optimizer='SPSA', exp_values=spsa_exp,
        iter_times=spsa_times, total_time=spsa_total))
    print(f'    final={spsa_exp[-1]:.4f}  min={min(spsa_exp):.4f}  time={spsa_total:.1f}s')

    del spsa_grad; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    # ── 3. CMA-ES ──
    print(f'  [3/3] CMA-ES ...')
    cma = CMAES(
        sigma=CMA_SIGMA, measure_method=MEASURE, shots=SHOTS)

    t0 = time.time()
    _, cma_exp, _, cma_times = minimize_cma_es(
        circuit, theta0.clone(), hamil, cma,
        generations=N_GENS, best_value_method='highest_probability')
    cma_total = time.time() - t0
    cma_exp = [float(v) for v in cma_exp]

    results.append(dict(
        ansatz=ansatz, n_qubits=nq, layers=layers, params=P,
        optimizer='CMA-ES', exp_values=cma_exp,
        iter_times=cma_times, total_time=cma_total))
    print(f'    final={cma_exp[-1]:.4f}  min={min(cma_exp):.4f}  time={cma_total:.1f}s')

    del cma; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'\nDone! {len(results)} runs completed.')

## 5. Build Results DataFrame

In [ ]:
records = []
for r in results:
    ev = r['exp_values']
    records.append({
        'ansatz': r['ansatz'],
        'n_qubits': r['n_qubits'],
        'layers': r['layers'],
        'params': r['params'],
        'optimizer': r['optimizer'],
        'final_E': ev[-1],
        'min_E': min(ev),
        'total_time_s': r['total_time'],
        'med_iter_ms': np.median(r['iter_times'][2:]) * 1000 if len(r['iter_times']) > 2 else np.nan,
        'evals_per_iter': 2 * r['params'] if r['optimizer'] == 'ParamShift'
                          else 2 if r['optimizer'] == 'SPSA'
                          else (4 + int(3 * math.log(r['params']))),  # CMA-ES pop size
    })

df = pd.DataFrame(records)
df['total_evals'] = df.apply(
    lambda row: row['evals_per_iter'] * len([r for r in results
        if r['optimizer'] == row['optimizer'] and r['ansatz'] == row['ansatz']
        and r['n_qubits'] == row['n_qubits'] and r['layers'] == row['layers']
    ][0]['exp_values']), axis=1)

display(df[['ansatz', 'n_qubits', 'layers', 'params', 'optimizer',
            'final_E', 'min_E', 'evals_per_iter', 'total_evals',
            'med_iter_ms', 'total_time_s']].round(3))

## 6. Convergence Plots

In [ ]:
COLORS = {
    'ParamShift': 'tab:blue',
    'SPSA':       'tab:orange',
    'CMA-ES':     'tab:green',
}

ansatze = ['QAOA', 'HEA']
nq_vals = sorted(df['n_qubits'].unique())
l_vals  = sorted(df['layers'].unique())

n_rows = len(ansatze)
n_cols = len(nq_vals) * len(l_vals)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5 * n_cols, 4 * n_rows),
                         squeeze=False, sharex=True)
fig.suptitle('Expectation Value Convergence by Iteration', fontsize=14, y=1.02)

for row, ansatz in enumerate(ansatze):
    col = 0
    for nq in nq_vals:
        for layers in l_vals:
            ax = axes[row][col]
            for r in results:
                if r['ansatz'] == ansatz and r['n_qubits'] == nq and r['layers'] == layers:
                    color = COLORS[r['optimizer']]
                    ax.plot(r['exp_values'], color=color,
                            label=r['optimizer'], alpha=0.85, linewidth=1.5)
            ax.set_title(f"{ansatz}  N={nq}  L={layers}  (P={2*nq*layers})", fontsize=10)
            if row == n_rows - 1:
                ax.set_xlabel('Iteration')
            if col == 0:
                ax.set_ylabel('$\\langle H \\rangle$')
            if row == 0 and col == 0:
                ax.legend(fontsize=8)
            col += 1

plt.tight_layout()
plt.show()

## 7. Convergence vs Wall-Clock Time

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5 * n_cols, 4 * n_rows),
                         squeeze=False)
fig.suptitle('Expectation Value vs Wall-Clock Time', fontsize=14, y=1.02)

for row, ansatz in enumerate(ansatze):
    col = 0
    for nq in nq_vals:
        for layers in l_vals:
            ax = axes[row][col]
            for r in results:
                if r['ansatz'] == ansatz and r['n_qubits'] == nq and r['layers'] == layers:
                    cum_time = np.cumsum(r['iter_times'])
                    color = COLORS[r['optimizer']]
                    ax.plot(cum_time, r['exp_values'], color=color,
                            label=r['optimizer'], alpha=0.85, linewidth=1.5)
            ax.set_title(f"{ansatz}  N={nq}  L={layers}", fontsize=10)
            if row == n_rows - 1:
                ax.set_xlabel('Time (s)')
            if col == 0:
                ax.set_ylabel('$\\langle H \\rangle$')
            if row == 0 and col == 0:
                ax.legend(fontsize=8)
            col += 1

plt.tight_layout()
plt.show()

## 8. Convergence vs Total Circuit Evaluations

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5 * n_cols, 4 * n_rows),
                         squeeze=False)
fig.suptitle('Expectation Value vs Total Circuit Evaluations', fontsize=14, y=1.02)

for row, ansatz in enumerate(ansatze):
    col = 0
    for nq in nq_vals:
        for layers in l_vals:
            ax = axes[row][col]
            for r in results:
                if r['ansatz'] == ansatz and r['n_qubits'] == nq and r['layers'] == layers:
                    P = r['params']
                    if r['optimizer'] == 'ParamShift':
                        evals_per = 2 * P
                    elif r['optimizer'] == 'SPSA':
                        evals_per = 2
                    else:  # CMA-ES
                        evals_per = 4 + int(3 * math.log(P))
                    n_steps = len(r['exp_values'])
                    cum_evals = np.arange(1, n_steps + 1) * evals_per
                    color = COLORS[r['optimizer']]
                    ax.plot(cum_evals, r['exp_values'], color=color,
                            label=f"{r['optimizer']} ({evals_per}/iter)",
                            alpha=0.85, linewidth=1.5)
            ax.set_title(f"{ansatz}  N={nq}  L={layers}  (P={2*nq*layers})", fontsize=10)
            if row == n_rows - 1:
                ax.set_xlabel('Circuit Evaluations')
            if col == 0:
                ax.set_ylabel('$\\langle H \\rangle$')
            ax.legend(fontsize=7)
            col += 1

plt.tight_layout()
plt.show()

## 9. Per-Iteration Timing

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, ansatz in enumerate(ansatze):
    ax = axes[i]
    sub = df[df['ansatz'] == ansatz].copy()
    sub['label'] = sub.apply(lambda r: f"N={r['n_qubits']} L={r['layers']}", axis=1)
    pivot = sub.pivot_table(index='label', columns='optimizer',
                            values='med_iter_ms', aggfunc='mean')
    pivot = pivot[['ParamShift', 'SPSA', 'CMA-ES']]  # fixed order
    pivot.plot.bar(ax=ax, color=[COLORS['ParamShift'], COLORS['SPSA'], COLORS['CMA-ES']],
                   edgecolor='black', width=0.7)
    ax.set_title(f'{ansatz}: Median Iteration Time', fontsize=12)
    ax.set_ylabel('Time (ms)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 10. Running-Best Expectation Value

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5 * n_cols, 4 * n_rows),
                         squeeze=False, sharex=True)
fig.suptitle('Running-Best $\\langle H \\rangle$ by Iteration', fontsize=14, y=1.02)

for row, ansatz in enumerate(ansatze):
    col = 0
    for nq in nq_vals:
        for layers in l_vals:
            ax = axes[row][col]
            for r in results:
                if r['ansatz'] == ansatz and r['n_qubits'] == nq and r['layers'] == layers:
                    running_best = np.minimum.accumulate(r['exp_values'])
                    color = COLORS[r['optimizer']]
                    ax.plot(running_best, color=color,
                            label=r['optimizer'], alpha=0.85, linewidth=1.5)
            ax.set_title(f"{ansatz}  N={nq}  L={layers}", fontsize=10)
            if row == n_rows - 1:
                ax.set_xlabel('Iteration')
            if col == 0:
                ax.set_ylabel('Best $\\langle H \\rangle$')
            if row == 0 and col == 0:
                ax.legend(fontsize=8)
            col += 1

plt.tight_layout()
plt.show()

## 11. Summary Table

In [ ]:
summary = df[['ansatz', 'n_qubits', 'layers', 'params', 'optimizer',
              'min_E', 'final_E', 'evals_per_iter', 'total_evals',
              'med_iter_ms', 'total_time_s']].copy()
summary = summary.round(3)

print('=== VQE Optimizer Comparison: Ring ZZ Hamiltonian ===')
print(f'Measurement: Efficient Contraction (exact, shots=0)')
print(f'Bond dimension: {RANK}')
print()

try:
    display(summary.style.background_gradient(
        subset=['min_E'], cmap='RdYlGn_r'
    ).background_gradient(
        subset=['total_time_s'], cmap='RdYlGn_r'
    ))
except:
    print(summary.to_string(index=False))

## 12. Speedup Analysis

In [ ]:
print('=== Per-iteration speedup over Parameter-Shift ===')
print()
for ansatz in ansatze:
    for nq in nq_vals:
        for layers in l_vals:
            sub = df[(df['ansatz'] == ansatz) & (df['n_qubits'] == nq) & (df['layers'] == layers)]
            t_ps = sub[sub['optimizer'] == 'ParamShift']['med_iter_ms'].values
            t_spsa = sub[sub['optimizer'] == 'SPSA']['med_iter_ms'].values
            t_cma = sub[sub['optimizer'] == 'CMA-ES']['med_iter_ms'].values
            if len(t_ps) and len(t_spsa) and len(t_cma):
                P = sub['params'].values[0]
                print(f'{ansatz} N={nq} L={layers} (P={P}):')
                print(f'  ParamShift: {t_ps[0]:.1f} ms/iter  (2P={2*P} evals/iter)')
                print(f'  SPSA:       {t_spsa[0]:.1f} ms/iter  (2 evals/iter)  '
                      f'-> {t_ps[0]/t_spsa[0]:.1f}x faster per iter')
                print(f'  CMA-ES:     {t_cma[0]:.1f} ms/iter  '
                      f'({4 + int(3*math.log(P))} evals/gen)')
                print()

---
# Part 2: Deep Circuits — CMA-ES vs Parameter-Shift

Stress-test with larger qubit counts, deeper layers, and higher bond dimension.
Does CMA-ES's O(n^2) covariance matrix start hurting at P=72, or does it still beat ParamShift's 2P evaluations per step?

## 13. Deep Circuit Configuration

In [ ]:
deep_configs = [
    # (n_qubits, layers, ansatz)
    (6,  4, 'QAOA'),   # P=48
    (6,  4, 'HEA'),    # P=48
    (8,  4, 'QAOA'),   # P=64
    (8,  4, 'HEA'),    # P=64
    (10, 3, 'QAOA'),   # P=60
    (10, 3, 'HEA'),    # P=60
    (12, 2, 'QAOA'),   # P=48
    (12, 2, 'HEA'),    # P=48
    (12, 3, 'QAOA'),   # P=72
    (12, 3, 'HEA'),    # P=72
    (14, 2, 'QAOA'),   # P=56
    (14, 2, 'HEA'),    # P=56
]

DEEP_ITERS = 300
DEEP_GENS  = 300
DEEP_RANK  = 16
DEEP_LR    = 0.03

print(f'{len(deep_configs)} configs x 2 optimizers = {len(deep_configs) * 2} runs')
print(f'Rank={DEEP_RANK}, Iters/Gens={DEEP_ITERS}, LR={DEEP_LR}')

## 14. Run Deep Experiments

In [ ]:
deep_results = []

for idx, (nq, layers, ansatz) in enumerate(deep_configs):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    if ansatz == 'QAOA':
        circuit = build_qaoa_circuit(nq, layers, DEEP_RANK)
    else:
        circuit = build_hea_circuit(nq, layers, DEEP_RANK)

    hamil = ring_zz_hamiltonian(nq)
    P = circuit.params_size
    theta0 = 0.1 * torch.randn(P, device=DEVICE,
        generator=torch.Generator(device=DEVICE).manual_seed(SEED))

    print(f'\n{"="*60}')
    print(f'[{idx+1}/{len(deep_configs)}] {ansatz} N={nq} L={layers} P={P} rank={DEEP_RANK}')
    print(f'{"="*60}')

    # ── ParamShift + Adam ──
    print(f'  ParamShift ...')
    ps_grad = BatchParameterShiftGradient(
        shift=math.pi / 2, batch_size=None, shots=0,
        measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=1)
    ps_opt = Optimizer(torch.optim.Adam, {'lr': DEEP_LR})

    t0 = time.time()
    _, ps_exp, _, ps_times = minimize(
        circuit, theta0.clone(), hamil, ps_opt, ps_grad,
        iteration=DEEP_ITERS, best_value_method='highest_probability')
    ps_total = time.time() - t0
    ps_exp = [float(v) for v in ps_exp]

    deep_results.append(dict(
        ansatz=ansatz, n_qubits=nq, layers=layers, params=P, rank=DEEP_RANK,
        optimizer='ParamShift', exp_values=ps_exp,
        iter_times=ps_times, total_time=ps_total))
    print(f'    min={min(ps_exp):.4f}  final={ps_exp[-1]:.4f}  time={ps_total:.1f}s')

    if hasattr(ps_grad, '_gpu_pool') and ps_grad._gpu_pool is not None:
        ps_grad._gpu_pool.shutdown()
    del ps_grad; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    # ── CMA-ES ──
    print(f'  CMA-ES ...')
    cma = CMAES(sigma=0.5, measure_method=MeasureMethod.EFFICIENT_CONTRACTION, shots=0)

    t0 = time.time()
    _, cma_exp, _, cma_times = minimize_cma_es(
        circuit, theta0.clone(), hamil, cma,
        generations=DEEP_GENS, best_value_method='highest_probability')
    cma_total = time.time() - t0
    cma_exp = [float(v) for v in cma_exp]

    deep_results.append(dict(
        ansatz=ansatz, n_qubits=nq, layers=layers, params=P, rank=DEEP_RANK,
        optimizer='CMA-ES', exp_values=cma_exp,
        iter_times=cma_times, total_time=cma_total))
    print(f'    min={min(cma_exp):.4f}  final={cma_exp[-1]:.4f}  time={cma_total:.1f}s')

    del cma; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'\nDone! {len(deep_results)} runs.')

## 15. Deep Results DataFrame

In [ ]:
deep_records = []
for r in deep_results:
    ev = r['exp_values']
    P = r['params']
    epi = 2 * P if r['optimizer'] == 'ParamShift' else (4 + int(3 * math.log(P)))
    deep_records.append({
        'ansatz': r['ansatz'], 'n_qubits': r['n_qubits'],
        'layers': r['layers'], 'params': P, 'rank': r['rank'],
        'optimizer': r['optimizer'],
        'final_E': ev[-1], 'min_E': min(ev),
        'evals_per_iter': epi,
        'total_evals': epi * len(ev),
        'med_iter_ms': np.median(r['iter_times'][2:]) * 1000,
        'total_time_s': r['total_time'],
    })

dfd = pd.DataFrame(deep_records)

display(dfd[['ansatz', 'n_qubits', 'layers', 'params', 'optimizer',
             'min_E', 'final_E', 'evals_per_iter', 'total_evals',
             'med_iter_ms', 'total_time_s']].round(3))

## 16. Deep Convergence by Iteration

In [ ]:
DEEP_COLORS = {'ParamShift': 'tab:blue', 'CMA-ES': 'tab:green'}

unique_configs = list(dict.fromkeys(
    [(r['ansatz'], r['n_qubits'], r['layers']) for r in deep_results]))
n_plots = len(unique_configs)
n_cols_d = 4
n_rows_d = math.ceil(n_plots / n_cols_d)

fig, axes = plt.subplots(n_rows_d, n_cols_d, figsize=(5*n_cols_d, 4*n_rows_d), squeeze=False)
fig.suptitle('Deep Circuits: Convergence by Iteration (rank=16)', fontsize=14, y=1.02)
for i, (ans, nq, L) in enumerate(unique_configs):
    ax = axes[i // n_cols_d][i % n_cols_d]
    for r in deep_results:
        if r['ansatz'] == ans and r['n_qubits'] == nq and r['layers'] == L:
            ax.plot(r['exp_values'], color=DEEP_COLORS[r['optimizer']],
                    label=r['optimizer'], alpha=0.85, linewidth=1.5)
    ax.axhline(-nq, color='red', ls=':', alpha=0.4, label=f'GS = {-nq}')
    ax.set_title(f"{ans} N={nq} L={L} (P={2*nq*L})", fontsize=9)
    ax.set_xlabel('Iteration')
    if i % n_cols_d == 0: ax.set_ylabel('$\\langle H \\rangle$')
    ax.legend(fontsize=7)
for j in range(i+1, n_rows_d*n_cols_d):
    axes[j // n_cols_d][j % n_cols_d].set_visible(False)
plt.tight_layout(); plt.show()

## 17. Deep Convergence vs Wall-Clock Time

In [ ]:
fig, axes = plt.subplots(n_rows_d, n_cols_d, figsize=(5*n_cols_d, 4*n_rows_d), squeeze=False)
fig.suptitle('Deep Circuits: Convergence vs Wall-Clock Time', fontsize=14, y=1.02)
for i, (ans, nq, L) in enumerate(unique_configs):
    ax = axes[i // n_cols_d][i % n_cols_d]
    for r in deep_results:
        if r['ansatz'] == ans and r['n_qubits'] == nq and r['layers'] == L:
            cum_t = np.cumsum(r['iter_times'])
            ax.plot(cum_t, r['exp_values'], color=DEEP_COLORS[r['optimizer']],
                    label=r['optimizer'], alpha=0.85, linewidth=1.5)
    ax.axhline(-nq, color='red', ls=':', alpha=0.4)
    ax.set_title(f"{ans} N={nq} L={L} (P={2*nq*L})", fontsize=9)
    ax.set_xlabel('Time (s)')
    if i % n_cols_d == 0: ax.set_ylabel('$\\langle H \\rangle$')
    ax.legend(fontsize=7)
for j in range(i+1, n_rows_d*n_cols_d):
    axes[j // n_cols_d][j % n_cols_d].set_visible(False)
plt.tight_layout(); plt.show()

## 18. Deep Convergence vs Circuit Evaluations

In [ ]:
fig, axes = plt.subplots(n_rows_d, n_cols_d, figsize=(5*n_cols_d, 4*n_rows_d), squeeze=False)
fig.suptitle('Deep Circuits: Convergence vs Circuit Evaluations', fontsize=14, y=1.02)
for i, (ans, nq, L) in enumerate(unique_configs):
    ax = axes[i // n_cols_d][i % n_cols_d]
    for r in deep_results:
        if r['ansatz'] == ans and r['n_qubits'] == nq and r['layers'] == L:
            P = r['params']
            epi = 2*P if r['optimizer'] == 'ParamShift' else (4+int(3*math.log(P)))
            cum_evals = np.arange(1, len(r['exp_values'])+1) * epi
            ax.plot(cum_evals, r['exp_values'], color=DEEP_COLORS[r['optimizer']],
                    label=f"{r['optimizer']} ({epi}/iter)", alpha=0.85, linewidth=1.5)
    ax.axhline(-nq, color='red', ls=':', alpha=0.4)
    ax.set_title(f"{ans} N={nq} L={L} (P={2*nq*L})", fontsize=9)
    ax.set_xlabel('Circuit Evaluations')
    if i % n_cols_d == 0: ax.set_ylabel('$\\langle H \\rangle$')
    ax.legend(fontsize=7)
for j in range(i+1, n_rows_d*n_cols_d):
    axes[j // n_cols_d][j % n_cols_d].set_visible(False)
plt.tight_layout(); plt.show()

## 19. Gap to Ground State

In [ ]:
print('=== Gap to Ground State (lower = better) ===')
print(f'{"Config":<25} {"ParamShift gap":>15} {"CMA-ES gap":>15} {"Winner":>10}')
print('-' * 70)
for ans, nq, L in unique_configs:
    sub = dfd[(dfd['ansatz']==ans) & (dfd['n_qubits']==nq) & (dfd['layers']==L)]
    gs = -nq
    ps_gap = sub[sub['optimizer']=='ParamShift']['min_E'].values[0] - gs
    cma_gap = sub[sub['optimizer']=='CMA-ES']['min_E'].values[0] - gs
    winner = 'ParamShift' if ps_gap < cma_gap else 'CMA-ES' if cma_gap < ps_gap else 'Tie'
    print(f'{ans} N={nq:>2} L={L} (P={2*nq*L:>3})  {ps_gap:>+15.6f} {cma_gap:>+15.6f} {winner:>10}')

## 20. Wall-Clock Speedup

In [ ]:
print('=== Wall-Clock Comparison (CMA-ES vs ParamShift) ===')
print()
for ans, nq, L in unique_configs:
    sub = dfd[(dfd['ansatz']==ans) & (dfd['n_qubits']==nq) & (dfd['layers']==L)]
    t_ps = sub[sub['optimizer']=='ParamShift']['total_time_s'].values[0]
    t_cma = sub[sub['optimizer']=='CMA-ES']['total_time_s'].values[0]
    P = sub['params'].values[0]
    print(f'{ans} N={nq:>2} L={L} (P={P:>3}): '
          f'PS={t_ps:.1f}s  CMA={t_cma:.1f}s  ratio={t_ps/t_cma:.2f}x')

## 21. Deep Summary Table

In [ ]:
print('=== Deep Circuit Summary: CMA-ES vs ParamShift ===')
print(f'Bond dimension: {DEEP_RANK}, Iterations: {DEEP_ITERS}')
print()

deep_summary = dfd[['ansatz', 'n_qubits', 'layers', 'params', 'optimizer',
                     'min_E', 'final_E', 'evals_per_iter', 'total_evals',
                     'med_iter_ms', 'total_time_s']].copy().round(3)

try:
    display(deep_summary.style.background_gradient(
        subset=['min_E'], cmap='RdYlGn_r'
    ).background_gradient(
        subset=['total_time_s'], cmap='RdYlGn_r'
    ))
except:
    print(deep_summary.to_string(index=False))